# ShellForge: full-data cloud GPU experiments
Run entirely in Colab. No model weights or training datasets need to be downloaded to your Mac.
Select **Runtime > Change runtime type > T4 GPU** (or another CUDA GPU), then **Run all**.
This notebook is an unexecuted workflow, not evidence of improved performance.
The original negative 20-step pilot, its adapter and original splits remain preserved.


In [ ]:
import subprocess, sys
from pathlib import Path
repo = Path('/content/ShellForge')
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', 'codex/review2-cloud',
                    'https://github.com/rupeshhh007/ShellForge.git', str(repo)], check=True)
%cd /content/ShellForge
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-model.txt', 'pypdf'], check=True)
subprocess.run(['apt-get', 'update'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'poppler-utils', 'fonts-dejavu-core'], check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), 'Connect a Colab CUDA GPU runtime before proceeding.'
print(torch.cuda.get_device_name(0))
print('GPU memory GiB:', torch.cuda.get_device_properties(0).total_memory / 2**30)


## Persist evidence in cloud storage
Mount Google Drive so logs, optimizer checkpoints and the final adapter survive a Colab disconnect.
The base model downloads into Colab's ephemeral cache. Run output stays in Drive.
A nonempty run folder is rejected to prevent overwriting experiments or accidental test-driven tuning.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
run_dir = Path('/content/drive/MyDrive/ShellForge/full-data-v1')


In [ ]:
subprocess.run([sys.executable, '-m', 'src.experiment_data', '--restore'], check=True)
test_log = repo / 'reports/logs/full_data_tests.log'
with test_log.open('w') as log:
    subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True, stdout=log, stderr=subprocess.STDOUT)
print(test_log.read_text())


## Bounded training and model selection
All quality-filtered training examples; all 1,255 validation examples for generation selection.
Proposed settings: 3 epochs maximum, rank 16, attention + MLP adapters, batch 2 x accumulation 8,
512-token supervised budget, cosine decay, 5% warmup, learning rates 1e-4 and 5e-5, seed 42.
FP16 on T4; BF16 only when supported. Ordinary LoRA fits this 0.5B model; QLoRA is optional.
Checkpoint every 100 steps; retain lowest validation-loss checkpoint, patience 3.
Compare candidates by validation strict exact match, tie-break on loss. Settings are frozen before test.
If a trial runs out of GPU memory, preserve its logs and reduce batch size while increasing accumulation
before any test has run. Do not edit settings based on test results.


In [ ]:
# Long-running cloud GPU job. Logs stream here and are saved under run_dir/logs.
subprocess.run([sys.executable, '-u', 'scripts/run_gpu_experiments.py',
                '--output', str(run_dir)], check=True)


## Inspect genuine results
The final comparison uses exactly the same 1,254 examples and greedy decoding for base and tuned.
The primary analysis contains the 1,222 examples not exposed in the historical 32-case pilot;
full-test results are also saved and explicitly disclose the 32 previously exposed examples.
Includes strict match, Wilson intervals, paired McNemar test, syntax validity, conservative AST trivia
normalization, AST node-kind overlap, category metrics, and separate safety classification.
AST metrics do not prove semantic equivalence. Review semantic_review.jsonl before claiming equivalence.
No generated command is ever executed. If validation does not improve, no improvement is claimed.


In [ ]:
import json
completion = json.loads((run_dir / 'completion.json').read_text())
comparison = json.loads((run_dir / 'comparison.json').read_text())
print(json.dumps(completion, indent=2))
print(json.dumps(comparison['primary_unexposed_test'], indent=2))
selection = json.loads((run_dir / 'selection.json').read_text())
adapter = selection['selected_adapter']
print('Adapter:', adapter)
subprocess.run([sys.executable, '-m', 'src.inference', 'Show the current working directory',
                '--backend', 'hf', '--revision', selection['revision'], '--adapter', adapter], check=True)


## Regenerate the four-page university report from saved evidence
The report keeps all six required sections and the negative pilot results. The builder refuses to use
incomplete runs and adds only measured full-data results. Report generation remains in Colab.
To continue analysis in Codex, share the small comparison/manifest/log files or a cloud link;
keep the model weights in Drive.


In [ ]:
subprocess.run([sys.executable, 'scripts/build_report.py', '--experiment', str(run_dir)], check=True)
import shutil
for name in ['ShellForge_Review2.pdf', 'ShellForge_Review2.md']:
    shutil.copy2(repo / 'reports' / name, run_dir / name)
from pypdf import PdfReader
pdf = PdfReader(str(run_dir / 'ShellForge_Review2.pdf'))
assert len(pdf.pages) == 4
assert all(abs(float(p.mediabox.width)-595.276)<1 and abs(float(p.mediabox.height)-841.89)<1 for p in pdf.pages)
subprocess.run(['pdftoppm', '-scale-to', '1200', '-png', str(run_dir / 'ShellForge_Review2.pdf'), str(run_dir / 'review')], check=True)
from IPython.display import display, IFrame
display(IFrame(str(repo / 'reports/ShellForge_Review2.pdf'), width=850, height=1050))
print('Cloud report:', run_dir / 'ShellForge_Review2.pdf')
